# Data Analysis (Part 2)
All Dataframes using Pandas, Polars and PySpark will be named by "df", "dp" and "data", respectively.

In [ ]:
# Import libraries
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pyspark
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import IntegerType
from pyspark.sql import functions as F

In [ ]:
# SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Local")
    .master("local[*]") # For local mode with all available cores
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("--- Successful PySpark Configuration ---")
print(f"Execution Mode: {spark.conf.get('spark.master')}")
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")

In [ ]:
# Create DataFrame from CSV file
df = pd.read_csv('../data/games.csv',sep=',',header=0)
dp = pl.read_csv('../data/games.csv',separator=',',has_header=True)
data = spark.read.csv('../data/games.csv',header=True,inferSchema=True)

## Replace

In [14]:
# Replace values
df['platform'] = df['platform'].replace({'PS4_old':'PlayStation4_new', 'XOne_old':'Xbox One_new'})

dp = dp.with_columns(
    pl.col("platform").replace(
        {"PS4_old": "PlayStation4_new", "XOne_old": "Xbox One_new"},
        default=pl.col("platform") # Mantiene los valores que no coinciden
    )
)

from pyspark.sql import functions as F
mapping = {'PS4_old': 'PlayStation4_new', 'XOne_old': 'Xbox One_new'}
data = data.replace(to_replace=mapping, subset=['platform'])

/tmp/ipykernel_60475/1610530161.py:5: DeprecationWarning: The `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
  pl.col("platform").replace(


## Format

In [15]:
def format_content(element):
	'''Function to format data'''
	element = element.strip().lower().replace(" ","_").replace(",","")
	element = element.replace(";","").replace(".","")
	return element

In [17]:
# Loop to change column names
col_names = []
for old_name in df.columns:
	new_name = format_content(old_name)
	col_names.append(new_name)

df.columns = col_names
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'sales', 'sales2'], dtype='object')


In [18]:
# Change column names
dp.columns = [format_content(col) for col in dp.columns]
print(dp.columns)

['name', 'platform', 'year_of_release', 'genre', 'sales', 'sales2']


In [19]:
data = data.toDF(*[format_content(col) for col in data.columns])
print(data.columns)

['name', 'platform', 'year_of_release', 'genre', 'sales', 'sales2']


In [ ]:
# Column operation
df['sales2'] = df['sales']*2

dp = dp.with_columns(
    (pl.col('sales') * 2).alias('sales2')
)
dp.head(3)

In [ ]:
data = data.withColumn('sales2', data['sales']*2)
data.show(3)